# So sánh FlexRanSplit và PlaceRAN

Notebook đọc trực tiếp các file `solution.csv` trong:

- `FlexRanSplitProblem/results`
- `PlaceRAN/results_pulp`

Mỗi network chỉ được so sánh khi có kết quả ở cả hai thư mục. `runtime` của FlexRanSplit là thời gian giải một mô hình ILP; `runtime` của PlaceRAN là tổng thời gian solver của ba stage.

In [ ]:
from pathlib import Path
import pandas as pd

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'FlexRanSplitProblem').is_dir() and (candidate / 'PlaceRAN').is_dir():
            return candidate
    raise FileNotFoundError('Không tìm thấy thư mục gốc Flex-split')

REPO_ROOT = find_repo_root(Path.cwd())
FLEX_RESULTS = REPO_ROOT / 'FlexRanSplitProblem' / 'results'
PLACERAN_RESULTS = REPO_ROOT / 'PlaceRAN' / 'results_pulp'

# Cho phép import khi notebook được mở từ thư mục gốc Flex-split.
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from FlexRanSplitProblem.compare_results import discover_results, summarize_solution

print('FlexRanSplit:', FLEX_RESULTS)
print('PlaceRAN    :', PLACERAN_RESULTS)

In [ ]:
flex_files = discover_results(FLEX_RESULTS, 'flexible')
placeran_files = discover_results(PLACERAN_RESULTS, 'placeran')
common_networks = sorted(set(flex_files) & set(placeran_files))

if not common_networks:
    raise ValueError('Không có network size nào có kết quả ở cả hai phương pháp')

rows = []
for network in common_networks:
    flex = summarize_solution(flex_files[network])
    placeran = summarize_solution(placeran_files[network])
    rows.append({
        'Network': network,
        'PlaceRAN runtime (s)': placeran['runtime'],
        'FlexRanSplit runtime (s)': flex['runtime'],
        'PlaceRAN DRC count': placeran['drc_count'],
        'FlexRanSplit DRC count': flex['drc_count'],
    })

comparison = pd.DataFrame(rows).sort_values('Network').reset_index(drop=True)
print('Network được so sánh:', common_networks)

## Bảng 1 - Thời gian chạy

In [ ]:
runtime_table = comparison[[
    'Network',
    'PlaceRAN runtime (s)',
    'FlexRanSplit runtime (s)',
]].copy()
runtime_table.style.hide(axis='index').format({
    'PlaceRAN runtime (s)': '{:.3f}',
    'FlexRanSplit runtime (s)': '{:.3f}',
}).set_caption('So sánh thời gian chạy (giây)')

## Bảng 2 - Số DRC được sử dụng

Số DRC là số giá trị `DRC` khác nhau trong nghiệm cuối cùng của network.

In [ ]:
drc_table = comparison[[
    'Network',
    'PlaceRAN DRC count',
    'FlexRanSplit DRC count',
]].copy()
drc_table.style.hide(axis='index').set_caption('So sánh số DRC được sử dụng')